## Import

In [1]:
from ultralytics import YOLO
import os
import torch

Save models

In [2]:
VERSION = "uecfoodpix_v1"

SAVE_DIR = f"../models/{VERSION}"
os.makedirs(SAVE_DIR, exist_ok=True)

model YOLOv8n-seg

In [3]:
model = YOLO("yolov8s-seg.pt")

training models

## Data Quality Check (Optional but Recommended)

In [4]:
from PIL import Image
import glob
from pathlib import Path
from collections import Counter

# Check for corrupted images
train_images = glob.glob("../data/dataset_yolo/images/train/*.jpg")
val_images = glob.glob("../data/dataset_yolo/images/val/*.jpg")

corrupted = []
for img_path in train_images + val_images:
    try:
        img = Image.open(img_path)
        img.verify()
    except Exception as e:
        corrupted.append((img_path, str(e)))

if corrupted:
    print(f"Found {len(corrupted)} corrupted images:")
    for path, error in corrupted[:10]:  # Show first 10
        print(f"  {path}: {error}")
else:
    print("✓ All images are valid")

# Analyze class distribution
label_train = glob.glob("../data/dataset_yolo/labels/train/*.txt")
class_counts = Counter()

for label_file in label_train:
    with open(label_file) as f:
        lines = f.readlines()
        for line in lines:
            class_id = int(line.split()[0])
            class_counts[class_id] += 1

print(f"\nClass distribution (top 20):")
for class_id, count in class_counts.most_common(20):
    print(f"  Class {class_id}: {count} instances")

print(f"\nUnderrepresented classes (< 20 instances):")
underrep = [(cid, cnt) for cid, cnt in class_counts.items() if cnt < 20]
print(f"  {len(underrep)} classes with < 20 instances")
for cid, cnt in underrep[:10]:
    print(f"  Class {cid}: {cnt} instances")

✓ All images are valid

Class distribution (top 20):
  Class 100: 2313 instances
  Class 101: 408 instances
  Class 35: 369 instances
  Class 0: 349 instances
  Class 22: 218 instances
  Class 86: 203 instances
  Class 11: 150 instances
  Class 5: 146 instances
  Class 95: 146 instances
  Class 16: 135 instances
  Class 6: 134 instances
  Class 67: 129 instances
  Class 91: 106 instances
  Class 18: 106 instances
  Class 48: 102 instances
  Class 4: 96 instances
  Class 90: 96 instances
  Class 21: 94 instances
  Class 9: 91 instances
  Class 26: 91 instances

Underrepresented classes (< 20 instances):
  0 classes with < 20 instances


## Key Changes Made to Improve Accuracy

**Model & Training Hyperparameters:**
- ✓ Upgraded from YOLOv8n-seg → YOLOv8s-seg (3.5x more parameters)
- ✓ Increased epochs: 5 → 100 (20x more training)
- ✓ Increased batch size: 8 → 16 (better gradient estimates)
- ✓ Reduced patience: 20 → 10 (avoid overfitting)
- ✓ Added learning rate parameters for better convergence

**Next Steps to Further Improve Accuracy:**

1. **Handle Corrupted Images** - Run the data quality check above to identify corrupted JPEGs. Remove or re-download corrupted images.

2. **Address Class Imbalance** - Classes with 0% precision (sushi, takoyaki, grilled eggplant) likely have:
   - Too few samples (< 20 instances)
   - Poor annotation quality
   - Ambiguous images
   
   Options:
   - Collect more samples for underrepresented classes
   - Merge similar classes
   - Use weighted loss (YOLO supports this)

3. **Increase Model Size** - If GPU memory allows:
   - Try YOLOv8m-seg (67M params) for better accuracy
   - Reduce imgsz to 416 if VRAM is limited

4. **Data Augmentation** - Ensure food.yaml has augmentation enabled:
   - Mosaic, mixup, flip, rotate, etc.

5. **Training Duration** - Monitor if model converges before 100 epochs. If it does, training is working but you may need better data.

In [5]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in the active notebook kernel. Select the .venv kernel with CUDA-enabled PyTorch."
    )

print(f"Using GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

gpu_device = 0

results = model.train(
    data="food.yaml",
    epochs=20,
    imgsz=384,
    batch=4,
    workers=6,
    patience=10,
    project="../runs/segment",
    name=VERSION,
    device=gpu_device,
    amp=True,
    half=True,
    lr0=0.01,
    lrf=0.01,
    cache=True,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=5,
    augment=True,
)


Using GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU Memory: 6.4 GB
Ultralytics 8.4.46  Python-3.10.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=food.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=uecfoodpix_v1-12, nbs=64, nms=False, op

## Evaluation

In [6]:
if 'results' not in locals():
    print("Warning: Training results not available. Using latest model from runs directory...")
    import glob
    run_dirs = sorted(glob.glob("../runs/segment/*/"), key=lambda x: os.path.getmtime(x), reverse=True)
    if run_dirs:
        best_weights = os.path.join(run_dirs[0], "weights", "best.pt")
    else:
        raise FileNotFoundError("No trained models found in ../runs/segment/")
else:
    best_weights = os.path.join(results.save_dir, "weights", "best.pt")

print(f"Evaluating: {best_weights}")

if not os.path.exists(best_weights):
    raise FileNotFoundError(f"Model weights not found at: {best_weights}")

model = YOLO(best_weights)
metrics = model.val(device=0)

print(f"Seg mAP50      : {metrics.seg.map50:.4f}")
print(f"Seg mAP50-95   : {metrics.seg.map:.4f}")
print(f"Precision      : {metrics.seg.mp:.4f}")
print(f"Recall         : {metrics.seg.mr:.4f}")

print(f"Box mAP50      : {metrics.box.map50:.4f}")
print(f"Box mAP50-95   : {metrics.box.map:.4f}")


Evaluating: Z:\code\fooSegmentation-master\notebook\runs\runs\segment\uecfoodpix_v1-12\weights\best.pt
Ultralytics 8.4.46  Python-3.10.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,819,074 parameters, 0 gradients, 40.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1401.7900.7 MB/s, size: 90.0 KB)
val: Scanning Z:\code\fooSegmentation-master\data\dataset_yolo\labels\val.cache... 1800 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1800/1800  0.0s
val: Z:\code\fooSegmentation-master\data\dataset_yolo\images\val\2902.jpg: corrupt JPEG restored and saved
val: Z:\code\fooSegmentation-master\data\dataset_yolo\images\val\99.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 113/113 8.2it/s 13.7s0.1s
                   all       1800       2720      0.575      0.615      0.62